<a href="https://colab.research.google.com/github/Shashini294/Statistical-Learning-e22294/blob/main/e22294_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



## 1. Deriving the Marginal Density

By the **Law of Total Probability**, we can find the marginal probability density function $p(x_i)$ by summing the joint probability of the observation $X_i = x_i$ and the latent cluster assignment $C_i$ over all possible cluster states $1, \dots, K$:

$$p(x_i) = \sum_{k=1}^{K} p(x_i, C_i = k)$$

Using the definition of conditional probability, we can rewrite the joint probability as $p(x_i, C_i = k) = P(C_i = k) \cdot p(x_i \mid C_i = k)$. Substituting the given prior cluster probabilities $P(C_i = k) = \phi_k$ and the conditional multivariate Gaussian component densities $X_i \mid C_i = k \sim \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$, we obtain:

$$p(x_i) = \sum_{k=1}^{K} \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$$

### Interpretation

This density is called a **Gaussian mixture density** because it mathematically "mixes" or combines $K$ distinct individual Gaussian component distributions into a single, comprehensive probability distribution over the continuous space $\mathbb{R}^d$. The parameters $\phi_k$ act as the mixing weights that govern how much each individual Gaussian sub-population contributes to the overall global density.

---

## 2. Deriving the Posterior Cluster Probability

For a fixed observation $x_i$, **Bayes' Rule** allows us to evaluate the probability that the point belongs to a specific cluster $k$ after observing its spatial coordinates:

$$P(C_i = k \mid X_i = x_i) = \frac{P(X_i = x_i \mid C_i = k) P(C_i = k)}{p(x_i)}$$

By substituting the law of total probability expansion for the marginal density $p(x_i)$ (derived in Part 1) into the denominator, we express it as:

$$P(C_i = k \mid X_i = x_i) = \frac{P(X_i = x_i \mid C_i = k) P(C_i = k)}{\sum_{j=1}^{K} P(X_i = x_i \mid C_i = j) P(C_i = j)}$$

Now, substituting the exact model components ($P(C_i = k) = \phi_k$ and the Gaussian PDFs), we define the cluster responsibility $\gamma_{ik}$:

$$\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$

### Interpretation

The quantity $\gamma_{ik}$ is interpreted as the **posterior probability of cluster membership** or the **responsibility** of cluster $k$ for data point $x_i$. It represents a true probability because it is naturally bounded between $0$ and $1$ ($\gamma_{ik} \in [0, 1]$), and for any given data point $x_i$, the responsibilities sum up to exactly $1$ across all $K$ clusters ($\sum_{k=1}^K \gamma_{ik} = 1$). It quantifies our updated belief about which cluster generated $x_i$ after looking at the data.

---

## 3. One-Hot Encoding of the Latent Cluster Variable

Let $Z_i = [Z_{i1}, Z_{i2}, \dots, Z_{iK}]^T$ be a one-hot encoded latent random vector where $Z_{ik} = 1$ if $C_i = k$, and $Z_{ik} = 0$ otherwise.

Because $Z_{ik}$ is a binary indicator variable taking values in $\{0, 1\}$, its conditional expectation is mathematically equivalent to the conditional probability of the indicator being equal to $1$:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = (1 \cdot P(Z_{ik} = 1 \mid X_i = x_i)) + (0 \cdot P(Z_{ik} = 0 \mid X_i = x_i))$$

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$

Applying this component-wise to the entire vector $Z_i$, we obtain:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i = x_i] \\ \mathbb{E}[Z_{i2} \mid X_i = x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i = x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

### Conclusion

This proves that the **soft cluster assignment** vector in a Gaussian mixture model is not an arbitrary heuristic choice; it is exactly the strict mathematical **conditional expectation** $\mathbb{E}[Z_i \mid X_i = x_i]$ of the latent structural cluster indicator vector given the observed data point.

---

## 4. From Soft Assignment to Hard Clustering

* **Soft Clustering:** Assigns a single observation fractionally to **all** $K$ clusters simultaneously based on its posterior distribution. For instance, a point resting near a structural boundary might be assigned $65\%$ to cluster 1, $30\%$ to cluster 2, and $5\%$ to cluster 3. This captures the inherent ambiguity and classification uncertainty of overlapping data distributions.
* **Hard Clustering:** Forces a definitive, deterministic assignment of a data point to exactly one cluster by selecting the component that yields the highest posterior probability:

$$\hat{C}_i = \arg\max_{1 \le k \le K} \gamma_{ik}$$



Unlike soft clustering, hard clustering strips away all localized geometric uncertainty, reducing the continuous vector of fractional assignments down to a single categorical index.

---

## 5. Conditional Expectation of the Observation Given the Cluster

Given that an observation is known to originate strictly from cluster $k$ ($C_i = k$), its spatial distribution is modeled purely by the multivariate normal distribution $\mathcal{N}(\mu_k, \Sigma_k)$. The mathematical expected value of a Gaussian random vector is its mean parameter:

$$\mathbb{E}[X_i \mid C_i = k] = \mu_k$$

Because this expectation equals the mean parameter $\mu_k$, it structurally acts as the geometric **center (or centroid)** of cluster $k$ in the feature space $\mathbb{R}^d$.

### Comparison of the Two Conditional Expectations

* $\mathbb{E}[Z_i \mid X_i = x_i]$ maps **from data space to cluster space**. It takes a known data coordinate $x_i$ and outputs the *fractional probabilities* pointing to which hidden clusters it most likely belongs to (the soft cluster membership).
* $\mathbb{E}[X_i \mid C_i = k]$ maps **from cluster space to data space**. It ignores individual data observations and defines the *expected coordinate location* of an idealized point generated natively from inside cluster $k$ (the mean location of the cluster).

---

## 6. The Complete-Data Likelihood

If the latent labels $z_i$ (or their equivalent one-hot indicator vectors $z_{ik}$) were completely visible and known, the joint complete-data likelihood for $n$ independent observations would be:

$$p(x_1, \dots, x_n, z_1, \dots, z_n) = \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

Taking the natural logarithm transforms the nested products into standard linear summations:

$$\ell_c = \ln \left( \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

$$\ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

### Optimization Advantage

If $z_{ik}$ were known, this expression would be incredibly easy to maximize because the clusters decouple completely. Instead of dealing with an interwoven sum-of-logs problem, the log-likelihood splits into independent optimization problems for each cluster. We could compute the parameters ($\phi_k, \mu_k, \Sigma_k$) in closed-form via direct Maximum Likelihood Estimation (MLE) using basic arithmetic counts, sample means, and sample covariances for each isolated group, without needing an iterative process.

---

## 7. The EM Interpretation

In practice, the true latent cluster indicators $Z_{ik}$ are hidden. The Expectation-Maximization (EM) algorithm circumvents this by taking the conditional expectation of the complete-data log-likelihood $\ell_c$ with respect to the posterior distribution of the latent variables, given the current parameter estimates:

$$Q = \mathbb{E}_{Z \mid X} [\ell_c] = \sum_{i=1}^{n} \sum_{k=1}^{K} \mathbb{E}[Z_{ik} \mid X_i = x_i] \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

Substituting the conditional expectation derived in Part 3 ($\mathbb{E}[Z_{ik} \mid X_i = x_i] = \gamma_{ik}$), we get the structural formula for the $Q$ function:

$$Q = \sum_{i=1}^{n} \sum_{k=1}^{K} \gamma_{ik} \left[ \ln \phi_k + \ln \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

### Interpretation of the E-step

The E-step can be interpreted as a **conditional update** of cluster membership probabilities because it leaves the physical parameters of the Gaussians untouched while updating our subjective belief about where each point belongs. It dynamically recalculates the responsibilities $\gamma_{ik}$ based on the conditional probability of the current cluster locations given the static coordinates of the dataset.

---

## 8. Parameter Updates (The M-Step)

Maximizing the surrogate objective function $Q$ with respect to the model parameters yields the following standard GMM updates:

$$N_k = \sum_{i=1}^{n} \gamma_{ik}, \quad \phi_k^{\text{new}} = \frac{N_k}{n}$$

$$\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik} x_i$$

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$

### Role of Responsibility as a Weight

In these updates, the responsibility $\gamma_{ik}$ acts strictly as a **fractional membership weight** of observation $x_i$ in cluster $k$. Instead of a point contributing binary values ($1$ if it belongs, $0$ if it doesn't) to compute updates, it contributes a continuous proportion.

* $N_k$ becomes the *effective, fractional number of points* assigned to cluster $k$.
* $\mu_k^{\text{new}}$ is a weighted average where points that have high compatibility with cluster $k$ exert a strong pull on its new center location.
* $\Sigma_k^{\text{new}}$ calculates dispersion softly by scaling each structural covariance step by its corresponding posterior responsibility weight.

---

## 9. Interpretation: Structural Synthesis Paragraph

Gaussian mixture clustering can be framed as an iterative, self-correcting process of **conditional updating**. The model structure dictates that the mixture weight $\phi_k$ represents the global *prior probability* of any given observation belonging to cluster $k$. When an empirical data point $x_i$ is evaluated, the conditional multivariate Gaussian density $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ measures how *compatible* that point's coordinates are with the physical shape and orientation of cluster $k$. Combining these elements via Bayes' theorem during the **E-step** computes the responsibility $\gamma_{ik}$, which represents the *posterior probability* of cluster membership after observing $x_i$. Collectively, these responsibilities assemble the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_i]$, capturing data classification uncertainty.

The **M-step** completes the cycle by conditionally updating the cluster parameters (weights, means, and covariances) using these exact posterior membership probabilities as fractional weights. Ultimately, Gaussian mixture clustering is not an arbitrary heuristic partitioning mechanism, but rather a mathematically formal architecture of **probabilistic clustering based on the conditional expectations of latent cluster membership variables**.